In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [16]:
organism = "homo_sapiens"
cell_source = "kidney"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Get sheet names

In [17]:
metadata = pd.read_json("../metadata.json")
sheet_names = [obj["sheet_name"] for obj in metadata["tables"]]

sheet_names

['1. PT',
 '2. LOH (DL)',
 '3. LOH (AL)',
 '4. CD',
 '5. Mono1',
 '6. Mono2',
 '7. B cells',
 '8. T cells',
 '9. Plasma1',
 '10. Plasma2',
 '11. Pericyte',
 '12. Fibroblast',
 '13. Myofibroblast',
 '14. Mast cells',
 '15. EC']

## Read in file

In [18]:
excel = pd.read_excel(table_name, sheet_name = sheet_names, skiprows = 0)

for e in list(excel.values()):
    print(e.columns)

Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj

In [19]:
cols = [
    {
        "sheet_name": sheet_name,
        "cols": {
            "avg_logFC": "logfc",
            "gene": "feature_name",
            "p_val": "p_val",
            "p_val_adj": "p_corr"
        }
    } for sheet_name in sheet_names]

In [20]:
for idx, sheet in enumerate(cols):
    sname = sheet["sheet_name"]
    print(sname)
    excel_name = excel
    if idx < 2:
        excel_name = excel
    # only keep the columns we want
    excel_name[sname] = excel_name[sname].loc[:,sheet["cols"].keys()].rename(columns=sheet["cols"])
    # add sheet name to the dataframe
    excel_name[sname]["sheet_name"] = sname

1. PT
2. LOH (DL)
3. LOH (AL)
4. CD
5. Mono1
6. Mono2
7. B cells
8. T cells
9. Plasma1
10. Plasma2
11. Pericyte
12. Fibroblast
13. Myofibroblast
14. Mast cells
15. EC


In [21]:
df = pd.concat(list(excel.values()))
df.head()

,logfc,feature_name,p_val,p_corr,sheet_name
0,1.958635,GPX3,6.780000e-106,1.390000e-101,1. PT
1,1.821094,CUBN,5.490000e-136,1.120000e-131,1. PT
2,1.764531,CDH6,9.110000e-154,1.870000e-149,1. PT
3,1.736607,LRP2,1.530000e-144,3.140000e-140,1. PT
4,1.673290,PDZK1IP1,1.010000e-133,2.060000e-129,1. PT


In [22]:
ct = {i: i.split('. ')[-1] for i in sheet_names}
df["cell_type_label"] = df["sheet_name"].map(ct)
df.head()

,logfc,feature_name,p_val,p_corr,sheet_name,cell_type_label
0,1.958635,GPX3,6.780000e-106,1.390000e-101,1. PT,PT
1,1.821094,CUBN,5.490000e-136,1.120000e-131,1. PT,PT
2,1.764531,CDH6,9.110000e-154,1.870000e-149,1. PT,PT
3,1.736607,LRP2,1.530000e-144,3.140000e-140,1. PT,PT
4,1.673290,PDZK1IP1,1.010000e-133,2.060000e-129,1. PT,PT


## Unfiltered

In [23]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["logfc"],
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'PT',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'GPX3',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'PT',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'GPX3',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': '1. PT',
   'source_metrics': {'p_corr': 1.39e-101, 'log_fc': 1.958634748}}}]

In [24]:
with open("evidnece_unfiltered_metrics.json", 'w') as f:
    json.dump(result, f, indent=4)

## Perform the filtering

In [8]:
col_pval = "p_val"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [9]:
min_mean = 100
max_pval = 1e-10
min_lfc = 1
max_gene_shares = 4
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [10]:
markers[col_ct].value_counts()

celltype
Plasma2          20
LOH (DL)         20
PT               20
LOH (AL)         20
Pericyte         20
EC               20
B cells          20
CD               20
Myofibroblast    20
Mono2            20
Fibroblast       20
Mono1            20
Mast cells       20
Cycling cells    19
Plasma1          15
T cells          14
Name: count, dtype: int64

In [11]:
if col_rank:
    print(markers.groupby(col_ct)[col_rank].mean().sort_values())

celltype
T cells          0.327500
Cycling cells    0.439368
PT               0.440950
LOH (DL)         0.463800
EC               0.509750
Pericyte         0.518850
LOH (AL)         0.576600
Plasma1          0.604867
B cells          0.643500
CD               0.645600
Mono2            0.657950
Plasma2          0.657950
Myofibroblast    0.688750
Fibroblast       0.703450
Mono1            0.819900
Mast cells       0.918000
Name: pct.1, dtype: float64


In [12]:
markers[col_ct].value_counts()

celltype
Plasma2          20
LOH (DL)         20
PT               20
LOH (AL)         20
Pericyte         20
EC               20
B cells          20
CD               20
Myofibroblast    20
Mono2            20
Fibroblast       20
Mono1            20
Mast cells       20
Cycling cells    19
Plasma1          15
T cells          14
Name: count, dtype: int64

## Convert to evidence json

In [16]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'T cells',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'CD96',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'T cells',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'CD96',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000153283'},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'T cells',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'TOX',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'T cells',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'TOX',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000198846'},
  'source': {'source_type': 'deg',


## Save evidence

In [17]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 